In [10]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import rand_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from scipy.spatial.distance import cdist
import time
import scipy.io

In [2]:
def norm_based(data):
    n, d = data.shape
    l = []
    for i in range(n):
        l.append(np.linalg.norm(data[i]))
    
    return l / np.sum(l)

In [3]:
def kmeans_cost(data, centers, labels):
    cost = 0.0
    for i in range(len(data)):
        distance = np.linalg.norm(data[i] - centers[labels[i]]) ** 2
        cost += distance
    return cost

In [4]:
def get_results_kmeans(coreset_size, n_clusters, traindata, data_name):
    results = []
    kmeans = KMeans(n_clusters= n_clusters, init="k-means++", random_state=42).fit(traindata)
    optimal_labels = kmeans.predict(traindata)
    cost = kmeans_cost(traindata, kmeans.cluster_centers_, optimal_labels)
    probabilities = norm_based(traindata)
    for ssize in coreset_size:
        tic = time.time()
        avg_cost = 0.0
        rand_index = 0.0
        for _ in range(5):
            indices = np.random.choice(traindata.shape[0], size=ssize, replace=False, p=probabilities)
            X_samples = traindata[indices]
            kmeans = KMeans(n_clusters=n_clusters, init="k-means++").fit(X_samples)
            labels = kmeans.predict(traindata)
            centers = kmeans.cluster_centers_
            avg_cost += kmeans_cost(traindata, centers, labels)
            rand_index += rand_score(optimal_labels, labels)
        rand_index /= 5
        avg_cost /= 5
        toc = time.time()
        reduction = ((traindata.shape[0] - X_samples.shape[0])/traindata.shape[0])*100
        error = (abs(avg_cost - cost)/cost)*100
        results.append({'Sampling Type': 'Norm Based Sampling',
                            'Coreset Size': X_samples.shape[0],
                            'Average Cost': avg_cost,
                            'Reduction in Data Size': reduction,
                            'Error': error,
                            'Avg Rand Index': rand_index,
                            'Data': data_name,
                            'Optimal Cost': cost,
                            'Avg Time': (toc - tic)/5,
                            'Clustering Algorithm': 'KMeans++'})
    return results

In [5]:
from torchvision import datasets, transforms

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=False, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=False, transform=transform)

X_train = train_dataset.data.numpy().reshape(-1, 28*28) / 255.0
Y_train = train_dataset.targets.numpy()
X_test = test_dataset.data.numpy().reshape(-1, 28*28) / 255.0
Y_test = test_dataset.targets.numpy()

/Users/shubhagarwal/anaconda3/envs/nlp/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Users/shubhagarwal/anaconda3/envs/nlp/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <E03EDA44-89AE-3115-9796-62BA9E0E2EDE> /Users/shubhagarwal/anaconda3/envs/NLP/lib/python3.11/site-packages/torchvision/image.so
  Expected in:     <D2077E4D-18BC-34B9-8A9B-1EF634A0F416> /Users/shubhagarwal/anaconda3/envs/NLP/lib/python3.11/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/Users/shubhagarwal/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update 

In [6]:
results = get_results_kmeans([100, 500, 1000, 2000, 5000, 7000, 10000, 20000], 10, X_train, 'FMNIST')

/Users/shubhagarwal/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/Users/shubhagarwal/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/Users/shubhagarwal/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/Users/shubhagarwal/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicit

In [7]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)

In [9]:
def kmedoids_cost(data, centroids, labels):
    dist = cdist(data, centroids, 'cityblock')
    total_cost = np.sum(dist[np.arange(len(data)), labels])
    return total_cost

In [8]:
def kmediod(data, weights, k, max_iterations=500):
    data = np.asarray(data)
    
    mins = data.min(axis=0)
    maxs = data.max(axis=0)
    centroids = np.random.rand(k, data.shape[1]) * (maxs - mins) + mins
    
    for _ in range(max_iterations):
        dist = cdist(data, centroids, 'cityblock')
        weighted_dist = dist * weights[:, np.newaxis]
        labels = np.argmin(weighted_dist, axis=1)
        
        for j in range(k):
            cluster = labels == j
            if weights[cluster].sum() > 0:
                centroids[j] = np.average(data[cluster], axis=0, weights=weights[cluster])
            else:
                centroids[j] = np.random.rand(1, data.shape[1]) * (maxs - mins) + mins
    
    return centroids

def predict(data, centroids):
    dist = cdist(data, centroids, 'cityblock')
    labels = np.argmin(dist, axis=1)
    return labels

In [11]:
def get_results_kmediod(coreset_size, n_clusters, traindata, data_name):
    results = []
    centers = kmediod(traindata, np.ones(traindata.shape[0]), n_clusters)
    optimal_labels = predict(traindata, centers)
    cost = kmedoids_cost(traindata, centers, optimal_labels)
    probabilities = norm_based(traindata)
    for ssize in coreset_size:
        tic = time.time()
        avg_cost = 0.0
        rand_index = 0.0
        for _ in range(5):
            indices = np.random.choice(traindata.shape[0], size=ssize, replace=False, p=probabilities)
            X_samples = traindata[indices]
            centers = kmediod(X_samples, np.ones(X_samples.shape[0]), n_clusters)
            labels = predict(traindata, centers)
            avg_cost += kmedoids_cost(traindata, centers, labels)
            rand_index += rand_score(optimal_labels, labels)
        rand_index /= 5
        avg_cost /= 5
        toc = time.time()
        reduction = ((traindata.shape[0] - X_samples.shape[0])/traindata.shape[0])*100
        error = (abs(avg_cost - cost)/cost)*100
        results.append({'Sampling Type': 'Norm Based Sampling',
                            'Coreset Size': X_samples.shape[0],
                            'Average Cost': avg_cost,
                            'Reduction in Data Size': reduction,
                            'Error': error,
                            'Avg Rand Index': rand_index,
                            'Data': data_name,
                            'Optimal Cost': cost,
                            'Avg Time': (toc - tic)/5,
                            'Clustering Algorithm': 'KMediod'})
    return results

In [16]:
X, y = make_blobs(n_samples=10000, centers=50, n_features=100, random_state=42)
results = get_results_kmediod([100, 500, 1000, 1500, 3500, 5000, 7000, 10000], 50, X, 'Synthetic')

In [18]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)

In [19]:
mat_data = scipy.io.loadmat('olivettifaces.mat')
traindata = mat_data['faces'].T
traindata = pd.DataFrame(traindata)
traindata.dropna()
traindata.drop_duplicates()
traindata.shape

(400, 4096)

In [21]:
results = get_results_kmediod([10, 20, 50, 70, 100], 10, traindata.values, 'Face')

In [22]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)